# Define root water uptake parameters

MIN3P-DP uses the Battaglia and Sands (1997) formulation of root water uptake, but I wasn't able to find an example of appropriate parameters for maize. Instead, compare to parameters for the S-shaped function described by van Genuchten (1987).

In [ ]:
from pathlib import Path
import s3fs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from byte_util import van_genuchten_vwc, battaglia_sands

In [ ]:
# Get list of target soil series
s3 = s3fs.S3FileSystem(anon=False)
s3_base_path = 's3://carbonplan-carbon-removal/ew-workflows-data/min3p'

soil_params_path = (f'{s3_base_path}/'
                    'input-data/processed-data/soil_physical_parameters.parquet')
soil_params = pd.read_parquet(soil_params_path)

all_sites = list(soil_params.index.get_level_values(0).unique())

root_params = pd.DataFrame(index=soil_params.index, columns=['satwlim', 'satwfield', 'rew0', 'p1'])

In [ ]:
from scipy.optimize import minimize

# Define matric potential range
h = 10**np.linspace(-3, 2, 1000)  # m, pressure head

horizon_colors = {"A": "#7A5C47", "B": "#B97A57", "C": "#CDBFAE"}

fig, ax = plt.subplots(2, 4, figsize=(14, 6), sharex=True, sharey=True)

site = 'Yolo'
for i, site in enumerate(all_sites):
    for horizon in ['A', 'B', 'C']:
        if (site == 'Yolo' and horizon == 'B') or \
            (site == 'Pullman' and horizon == 'C'):
            continue

        # Convert matric potential to saturation
        theta_s = soil_params.loc[(site, horizon), 'theta_s']
        theta_r = soil_params.loc[(site, horizon), 'theta_r']
        alpha = soil_params.loc[(site, horizon), 'alpha_m']
        n = soil_params.loc[(site, horizon), 'n']

        # Calculate water content
        vwc = van_genuchten_vwc(-h, theta_r=theta_r, theta_s=theta_s, alpha=alpha, n=n)
        relsat = (vwc - theta_r) / (theta_s - theta_r)
        abssat = relsat * (1 - theta_r) + theta_r  # Convert relative saturation to absolute saturation

        ## Define van Genuchten (1987) parameters for maize
        # Using parameters from Wang et al. (2015) -- doi:10.1002/ird.1939
        h50 = 1.726  # In m
        p = 1.64
        rootwat_vg = 1 / (1 + (h/h50)**p)

        ## Fit Battaglia and Sands (1997) parameters
        def rmse(params):
            satwlim, satwfield, rew0, p1 = params
            rw_bs = battaglia_sands(abssat, satwlim, satwfield, rew0, p1)
            return np.sum((rw_bs - rootwat_vg) ** 2)

        # Bounds for each parameter
        bounds = [(theta_r/theta_s+0.01, 0.8),  # satwlim
                  (0.95, 1.), # satwfield
                  (0.0, 1.0), # rew0
                  (0.1, 5)]   # p1
        initial_guess = (0.12, 1.0, 0.14, 0.5)
        result = minimize(rmse, initial_guess, bounds=bounds, method="SLSQP")

        sat = np.linspace(0, 1, 1000)
        fit_rwbs = battaglia_sands(sat, *result.x)

        ax.flatten()[i].plot(sat, fit_rwbs, color=horizon_colors[horizon], ls='--')
        ax.flatten()[i].plot(abssat, rootwat_vg, color=horizon_colors[horizon], label=horizon)
        ax.flatten()[i].set_title(f'{site}: BS97 fit (--) vs W15 (-)', fontsize=10)
        ax.flatten()[i].legend()

        # Store fit parameters
        root_params.loc[(site, horizon)] = result.x

for i in range(2):
    ax[i, 0].set(ylabel='Relative root water uptake')
for i in range(4):
    ax[1, i].set(xlabel='Soil saturation')

# Save root water uptake parameters to file
root_params.to_parquet(f'{s3_base_path}/input-data/processed-data/root_water_parameters.parquet', index=True)

In [ ]:
# Define domain
nz = 401
max_depth = 4.0  # m
dz = max_depth / (nz - 1)

# Define vertical root distribution according to Gale and Grigal (1987) and
# parameters from Jackson et al. (1996) for "crops"
fig, ax = plt.subplots(1, 2, figsize=(8, 6), sharey=True)

beta_j96 = 0.961  # Jackson et al. (1996) for "crops"
beta_s16 = np.mean((0.92, 0.94, 0.96, 0.96, 0.96, 0.94))

z = np.linspace(0, max_depth*100, nz)  # in cm

for label, beta in {'J96': beta_j96, 'S16': beta_s16}.items():
    # Calculate cumulative root length density with depth
    y = 1 - beta**z

    # Take the derivative to get the root length density
    rld = -np.log(beta)*beta**z

    ax[0].plot(y, z, label=label)
    ax[1].plot(rld, z)
    ax[0].set(xlim=[0, 1], ylim=[200, 0], ylabel='Depth (cm)', xlabel='Cumulative fraction')
    ax[1].set(xlabel='Normalized root density')
ax[0].legend()

# Let's use Jackson (1996), since it better represents the deeper distribution used in P-M calculations
rld = -np.log(beta_j96)*beta_j96**z

# Normalize rld to sum to 1 and save to file
rld = rld / np.sum(rld)

flipped_rld = rld[::-1]  # Flip so that bottom = 0.0 m and surface = 4.0 m
base_path = Path('../simulations/met_forcing_transport/min3p_runs/base')
singleperm_path = base_path / 'singleperm.rld'
base_path = Path('../simulations/dual_perm_rxn/min3p_runs/base')
dualperm_path = base_path / 'dualperm.rld'
with open(singleperm_path, 'w') as f_singleperm:
    with open(dualperm_path, 'w') as f_dualperm:
        f_singleperm.write('variables = "x", "y", "z", "rld"\n')
        f_dualperm.write('variables = "x", "y", "z", "rld"\n')
        for i, zval in enumerate(z):
            zval_m = zval / 100  # Convert to m
            f_singleperm.write(f'  0.0000000E+00  0.0000000E+00  {zval_m:.7E}  {flipped_rld[i]:.7E}\n')
            # For dual permeability, assume no roots in macropores (y = 0)
            f_dualperm.write(f'  0.0000000E+00  0.0000000E+00  {zval_m:.7E}  0.0000000E+00\n')
            # But there are roots in the matrix domain (y = 1)
            f_dualperm.write(f'  0.0000000E+00  0.1000000E+01  {zval_m:.7E}  {flipped_rld[i]:.7E}\n')
        # For dualperm, add the distribution layer (1-cm thick) with no roots (y = 0)
        f_dualperm.write(f'  0.0000000E+00  0.0000000E+00  {zval_m+dz:.7E}  0.0000000E+00\n')
        f_dualperm.write(f'  0.0000000E+00  0.1000000E+01  {zval_m+dz:.7E}  0.0000000E+00\n')